# LangGraph — State Machines for Agents Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: state and nodes

In [ ]:
```python

from typing import Annotated, TypedDict

from langchain_core.messages import AnyMessage, HumanMessage, AIMessage

from langgraph.graph import StateGraph, END

from langgraph.graph.message import add_messages

from langgraph.prebuilt import ToolNode

from langgraph.checkpoint.memory import MemorySaver

class State(TypedDict):

    messages: Annotated[list[AnyMessage], add_messages]

def agent_node(state: State) -> dict:

    response = llm.invoke(state["messages"])

    return {"messages": [response]}

def should_continue(state: State) -> str:

    last = state["messages"][-1]

    return "tools" if getattr(last, "tool_calls", None) else END

tool_node = ToolNode(tools=[search_web, read_file])

graph = StateGraph(State)

graph.add_node("agent", agent_node)

graph.add_node("tools", tool_node)

graph.set_entry_point("agent")

graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})

graph.add_edge("tools", "agent")

app = graph.compile(checkpointer=MemorySaver())

In [ ]:
```

`add_messages` is the reducer that makes the message list accumulate instead of overwrite. Forgetting it is the most common LangGraph bug.

### Step 2: run with a thread

In [ ]:
```python

config = {"configurable": {"thread_id": "user-42"}}

for event in app.stream(

    {"messages": [HumanMessage("find the Anthropic headquarters address")]},

    config,

    stream_mode="updates",

):

    print(event)

In [ ]:
```

Every update is a dict `{node_name: state_delta}`. Your frontend can stream these to the UI so users see "agent is thinking… calling search_web… got result… answering."

### Step 3: add a human-in-the-loop interrupt

Mark a node so execution pauses before it runs.

In [ ]:
```python

app = graph.compile(

    checkpointer=MemorySaver(),

    interrupt_before=["tools"],  # pause before every tool call

)

state = app.invoke({"messages": [HumanMessage("delete the production database")]}, config)

# state["__interrupt__"] is set. Inspect proposed tool calls.

# If approved:

from langgraph.types import Command

app.invoke(Command(resume=True), config)

# If denied: write a rejection message and resume

app.update_state(config, {"messages": [AIMessage("Blocked by human reviewer.")]})

In [ ]:
```

The state, the checkpoint, and the thread all persist across the interrupt. Nothing is in memory except during execution.

### Step 4: time-travel for debugging

In [ ]:
```python

history = list(app.get_state_history(config))

for snapshot in history:

    print(snapshot.values["messages"][-1].content[:80], snapshot.config)

# Fork from a prior checkpoint

target = history[3].config  # three steps back

for event in app.stream(None, target, stream_mode="values"):

    pass  # replay from that point forward

In [ ]:
```

Passing `None` as the input replays from the given checkpoint; passing a value appends it as an update to that checkpoint's state before resuming. This is how you reproduce a bad agent run without re-running the whole conversation.

### Step 5: swap the checkpointer for production

In [ ]:
```python

from langgraph.checkpoint.postgres import PostgresSaver

with PostgresSaver.from_conn_string("postgresql://...") as checkpointer:

    checkpointer.setup()

    app = graph.compile(checkpointer=checkpointer)

In [ ]:
```

SQLite, Redis, and Postgres are shipped. `MemorySaver` is for tests. Anything that persists across restarts wants a real store.

## Exercises